In [ ]:
# -*- coding: utf-8 -*-
"""
Assumptions:
1. Raster filenames match feature names, e.g. slope.tif, aspect.tif, curvature.tif, etc.
2. Training models were saved by grid in:
   /content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/model_outputs/<grid>/models/
3. Excel files are the same ones used for training.
4. Features used for raster prediction are exactly the same as training features after dropping:
   ID, ndwi, roughness, X, Y
"""

!pip install openpyxl joblib xgboost rasterio scikit-learn -q

import os
import gc
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from google.colab import drive
from sklearn.covariance import LedoitWolf
from sklearn.model_selection import StratifiedGroupKFold

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

mpl.rcParams.update({
    "font.family":         "DejaVu Sans",
    "font.size":           11,
    "axes.titlesize":      12,
    "axes.titleweight":    "bold",
    "axes.labelsize":      11,
    "axes.linewidth":      1.0,
    "axes.spines.top":     False,
    "axes.spines.right":   False,
    "xtick.labelsize":     10,
    "ytick.labelsize":     10,
    "legend.fontsize":     10,
    "legend.frameon":      True,
    "legend.framealpha":   0.9,
    "legend.edgecolor":    "0.5",
    "figure.dpi":          120,
    "savefig.dpi":         600,
    "savefig.bbox":        "tight",
    "savefig.pad_inches":  0.05,
    "pdf.fonttype":        42,
    "ps.fonttype":         42,
})

MODEL_COLOURS = {
    "RF":       "#2E86AB",
    "SVM":      "#E07A5F",
    "XGB":      "#7E4E90",
    "Ensemble": "#0E1B2C",
}

PRETTY_FEATURE = {
    "aspect":         "Aspect",
    "curvature":      "Curvature (profile)",
    "dis_to_fault":   "Distance to fault",
    "dis_to_river":   "Distance to river",
    "dis_to_road":    "Distance to road",
    "dis_to_thrust":  "Distance to thrust",
    "ndvi":           "NDVI",
    "slope":          "Slope",
    "twi":            "TWI",
    "aspect_average":        "Aspect (mean)",
    "aspect_stddev":         "Aspect (sd)",
    "curvature_average":     "Curvature (mean)",
    "curvature_stddev":      "Curvature (sd)",
    "dis_to_fault_average":  "Dist. to fault (mean)",
    "dis_to_fault_stddev":   "Dist. to fault (sd)",
    "dis_to_river_average":  "Dist. to river (mean)",
    "dis_to_river_stddev":   "Dist. to river (sd)",
    "dis_to_road_average":   "Dist. to road (mean)",
    "dis_to_road_stddev":    "Dist. to road (sd)",
    "dis_to_thrust_average": "Dist. to thrust (mean)",
    "dis_to_thrust_stddev":  "Dist. to thrust (sd)",
    "ndvi_average":          "NDVI (mean)",
    "ndvi_stddev":           "NDVI (sd)",
    "slope_average":         "Slope (mean)",
    "slope_stddev":          "Slope (sd)",
    "twi_average":           "TWI (mean)",
    "twi_stddev":            "TWI (sd)",
}

def pretty(name):
    return PRETTY_FEATURE.get(name, name)


In [ ]:
# =========================================================
# 1. PATHS
# =========================================================
excel_path = "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/Filtered_Excel files/"
excel_files = {
    # "3x3": "ls_non_ls_final_3x3",
    "15x15": "ls_non_ls_final_15x15",
    "25x25": "ls_non_ls_final_25x25"
}

# folder for saved models
model_base_dir = "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/model_outputs_spatial"

# Each grid folder should contain rasters named exactly like feature names
raster_base_dirs = {
    # "3x3": "/content/drive/MyDrive/3x3_rasters",
    "15x15": "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/15x15_rasters",
    "25x25": "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/25x25_rasters"
}

output_base_dir = "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/susceptibility_outputs"
os.makedirs(output_base_dir, exist_ok=True)

# =========================================================
# 2. HELPERS
# =========================================================
def make_dir(path):
    os.makedirs(path, exist_ok=True)

def load_training_features(grid, feature_names, ensemble_bundle=None):
    """Load ONLY the training fold (spatially-blocked 80/20) the comparison
    notebook actually fit the models on.

    The applicability-domain covariance must not see the held-out test fold's
    pixels. Recreates the deterministic StratifiedGroupKFold first-fold split
    using the block size stored in the ensemble bundle (1000 m by default).
    """
    df = pd.read_excel(f"{excel_path}{excel_files[grid]}.xlsx")
    df.columns = df.columns.str.strip()
    df = df.dropna().reset_index(drop=True)

    y = df["ID"].astype(int).values
    X_full = df[feature_names].copy()

    # If we don't have a bundle (legacy path), fall back to the old behaviour.
    if ensemble_bundle is None or "X" not in df.columns or "Y" not in df.columns:
        return X_full, y

    block_size = int(ensemble_bundle.get("spatial_block_size", 1000))
    bx = np.floor(df["X"].values / block_size).astype(int)
    by = np.floor(df["Y"].values / block_size).astype(int)
    groups = pd.Series(
        pd.Series(bx).astype(str) + "_" + pd.Series(by).astype(str)
    )
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    train_idx, _ = next(splitter.split(X_full, y, groups=groups))
    X_train = X_full.iloc[train_idx].reset_index(drop=True)
    y_train = y[train_idx]
    print(f"  AD fit on training fold only: {len(X_train)} of {len(X_full)} "
          f"samples (block={block_size} m)")
    return X_train, y_train

def load_models_for_grid(grid):
    model_dir = os.path.join(model_base_dir, grid, "models")

    rf = joblib.load(os.path.join(model_dir, f"RF_{grid}_calibrated.pkl"))
    svm = joblib.load(os.path.join(model_dir, f"SVM_{grid}_calibrated.pkl"))
    xgb = joblib.load(os.path.join(model_dir, f"XGB_{grid}_calibrated.pkl"))

    ensemble_bundle = joblib.load(os.path.join(model_dir, f"Ensemble_{grid}.pkl"))

    return rf, svm, xgb, ensemble_bundle

def load_raster_stack(grid, feature_names):
    raster_dir = raster_base_dirs[grid]

    arrays = []
    raster_paths = {}
    profile = None
    transform = None
    crs = None
    nodata_ref = None
    bounds = None

    for i, feature in enumerate(feature_names):
        path = os.path.join(raster_dir, f"{feature}_{grid}.tif")
        if not os.path.exists(path):
            raise FileNotFoundError(f"Missing raster for feature '{feature}': {path}")

        raster_paths[feature] = path

        with rasterio.open(path) as src:
            arr = src.read(1).astype(np.float32)

            if i == 0:
                profile = src.profile.copy()
                transform = src.transform
                crs = src.crs
                nodata_ref = src.nodata
                bounds = src.bounds          
                ref_shape = arr.shape
            else:
                if arr.shape != ref_shape:
                    raise ValueError(f"Raster shape mismatch for feature '{feature}'. Expected {ref_shape}, got {arr.shape}")

            arrays.append(arr)

    stack = np.stack(arrays, axis=0)   # [bands, rows, cols]
    return stack, profile, transform, crs, nodata_ref, raster_paths, bounds

def build_valid_mask(stack, nodata_ref=None):
    valid = np.all(np.isfinite(stack), axis=0)

    if nodata_ref is not None:
        valid &= np.all(stack != nodata_ref, axis=0)

    return valid

def stack_to_dataframe(stack, valid_mask, feature_names):
    bands, rows, cols = stack.shape
    flat = stack.reshape(bands, -1).T
    flat_valid = flat[valid_mask.ravel()]
    df = pd.DataFrame(flat_valid, columns=feature_names)
    return df

def write_raster(path, array2d, profile, nodata_value=-9999.0):
    out_profile = profile.copy()
    out_profile.update(
        dtype=rasterio.float32,
        count=1,
        compress="lzw",
        nodata=nodata_value
    )

    out_arr = np.where(np.isfinite(array2d), array2d, nodata_value).astype(np.float32)

    # Drive FUSE on Colab refuses rasterio's Create()-then-overwrite when the
    # target TIFF already exists, returning CPLE_OpenFailedError. Write to a
    # local temp file first and then move into the target path (unlink+rename
    # both work under FUSE).
    import tempfile as _tempfile
    import shutil as _shutil
    _fd, _tmp = _tempfile.mkstemp(suffix=".tif", prefix="raster_")
    os.close(_fd)
    try:
        with rasterio.open(_tmp, "w", **out_profile) as dst:
            dst.write(out_arr, 1)
        if os.path.exists(path):
            try: os.remove(path)
            except OSError: pass
        _shutil.move(_tmp, path)
    finally:
        if os.path.exists(_tmp):
            try: os.remove(_tmp)
            except OSError: pass

def _add_carto(ax, bounds, scale_km=5, crs_label="UTM Zone 44N (EPSG:32644)"):
    """Add scale bar + north arrow + CRS tag + UTM-in-km axes to a map axis."""
    bar_len_m = scale_km * 1000
    map_width = bounds.right - bounds.left
    bar_frac = (bar_len_m / map_width) * 0.95

    bx0 = 0.78
    bx1 = min(bx0 + bar_frac, 0.97)
    ax.plot([bx0, bx1], [0.04, 0.04], color="black", linewidth=4,
            transform=ax.transAxes, solid_capstyle="butt", zorder=10)
    ax.plot([bx0, bx0], [0.028, 0.052], color="black", linewidth=1.5,
            transform=ax.transAxes, zorder=10)
    ax.plot([bx1, bx1], [0.028, 0.052], color="black", linewidth=1.5,
            transform=ax.transAxes, zorder=10)
    ax.text(0.5 * (bx0 + bx1), 0.060, f"{scale_km} km",
            transform=ax.transAxes, ha="center", va="bottom", fontsize=9,
            bbox=dict(facecolor="white", edgecolor="none", pad=1, alpha=0.85))

    ax.annotate("", xy=(0.95, 0.92), xytext=(0.95, 0.86),
                xycoords="axes fraction",
                arrowprops=dict(arrowstyle="-|>", color="black", lw=1.6, mutation_scale=18),
                zorder=10)
    ax.text(0.95, 0.932, "N", transform=ax.transAxes, ha="center", va="bottom",
            fontsize=11, fontweight="bold", zorder=10,
            bbox=dict(facecolor="white", edgecolor="none", pad=1, alpha=0.85))

    ax.text(0.02, 0.97, crs_label, transform=ax.transAxes, ha="left", va="top",
            fontsize=8, zorder=10,
            bbox=dict(facecolor="white", edgecolor="0.5", pad=2, alpha=0.85))

    xt = np.linspace(bounds.left, bounds.right, 5)
    yt = np.linspace(bounds.bottom, bounds.top, 5)
    ax.set_xticks(xt); ax.set_yticks(yt)
    ax.set_xticklabels([f"{int(round(t/1000))}" for t in xt], fontsize=8)
    ax.set_yticklabels([f"{int(round(t/1000))}" for t in yt], fontsize=8)
    ax.set_xlabel("Easting (km, UTM 44N)", fontsize=9)
    ax.set_ylabel("Northing (km, UTM 44N)", fontsize=9)


def quick_plot(array2d, title, save_path, cmap="RdYlBu_r",
               bounds=None, vmin=0, vmax=1, scale_km=5,
               classified=False, class_labels=None):
    fig, ax = plt.subplots(figsize=(9, 6.5))
    if bounds is not None:
        extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
        im = ax.imshow(array2d, cmap=cmap, vmin=vmin, vmax=vmax,
                       extent=extent, origin="upper")
        _add_carto(ax, bounds, scale_km=scale_km)
    else:
        im = ax.imshow(array2d, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_xticks([]); ax.set_yticks([])
    cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.04)
    cb.set_label(title)
    if classified and class_labels:
        cb.set_ticks(np.arange(len(class_labels)) + 0.5)
        cb.set_ticklabels(class_labels)
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(save_path)
    plt.show()
    plt.close(fig)


def plot_multi_model_maps(maps_dict, bounds, grid_name, save_path,
                          cmap="RdYlBu_r", vmin=0, vmax=1):
    """2x2 composite of model-wise susceptibility maps."""
    fig, axes = plt.subplots(2, 2, figsize=(12, 11))
    extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
    last_im = None
    for ax, (name, arr) in zip(axes.flat, maps_dict.items()):
        last_im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax,
                            extent=extent, origin="upper")
        ax.set_title(f"{name} susceptibility")
        _add_carto(ax, bounds, scale_km=5)
    cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
    fig.colorbar(last_im, cax=cbar_ax, label="Landslide susceptibility probability")
    fig.suptitle(f"Landslide susceptibility maps - {grid_name}", y=1.00)
    fig.subplots_adjust(right=0.90, hspace=0.22, wspace=0.18)
    fig.savefig(save_path)
    plt.show()
    plt.close(fig)


def plot_classified_susceptibility(arr, bounds, grid_name, save_path,
                                   pixel_area_m2, n_classes=5, sample_size=10_000):
    """Jenks 5-class classified susceptibility map + areal-statistics table."""
    try:
        import jenkspy
    except ImportError:
        import subprocess as _sp, sys as _sys
        _sp.check_call([_sys.executable, "-m", "pip", "install", "-q", "jenkspy"])
        import jenkspy

    rng = np.random.default_rng(42)
    valid = arr[np.isfinite(arr)]
    sample = rng.choice(valid, size=min(sample_size, len(valid)), replace=False)
    raw = jenkspy.jenks_breaks(sample.tolist(), n_classes=n_classes)
    internal = sorted(raw)[1:-1]
    breaks = [0.0] + internal + [1.0]

    classed = np.full_like(arr, np.nan, dtype=np.float32)
    for k in range(n_classes):
        lo, hi = breaks[k], breaks[k + 1]
        if k == n_classes - 1:
            mask = np.isfinite(arr) & (arr >= lo)
        else:
            mask = np.isfinite(arr) & (arr >= lo) & (arr < hi)
        classed[mask] = k

    colours = ["#1A9850", "#A6D96A", "#FFFFBF", "#FDAE61", "#D7191C"]
    labels  = ["Very Low", "Low", "Moderate", "High", "Very High"]
    cmap = mpl.colors.ListedColormap(colours)
    norm = mpl.colors.BoundaryNorm(list(range(n_classes + 1)), cmap.N)

    fig, ax = plt.subplots(figsize=(11, 7.5))
    extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
    im = ax.imshow(classed, cmap=cmap, norm=norm, extent=extent, origin="upper")
    ax.set_title(f"Ensemble susceptibility (Jenks 5-class) - {grid_name}")

    cb = fig.colorbar(im, ax=ax, ticks=[0.5, 1.5, 2.5, 3.5, 4.5], pad=0.04)
    cb.ax.set_yticklabels(labels)
    cb.set_label("Susceptibility class")

    total = np.isfinite(arr).sum()
    text_lines = ["Areal statistics:"]
    rows = []
    for k in range(n_classes):
        count = int((classed == k).sum())
        pct = 100.0 * count / total if total else 0.0
        area_km2 = count * pixel_area_m2 / 1_000_000
        text_lines.append(f"{labels[k]:<10}{pct:>6.1f}%  ({area_km2:>7.2f} km2)")
        rows.append({"class": labels[k], "lower": breaks[k], "upper": breaks[k+1],
                     "pixels": count, "area_km2": round(area_km2, 2),
                     "percent": round(pct, 2)})
    ax.text(0.02, 0.20, "\n".join(text_lines), transform=ax.transAxes,
            fontsize=9, family="DejaVu Sans Mono", va="bottom",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="0.5", alpha=0.9))

    _add_carto(ax, bounds, scale_km=5)
    fig.tight_layout()
    fig.savefig(save_path)
    plt.show()
    plt.close(fig)

    return pd.DataFrame(rows)

def reshape_predictions(pred_valid, valid_mask, shape, fill_value=np.nan):
    out = np.full(shape, fill_value, dtype=np.float32)
    out[valid_mask] = pred_valid.astype(np.float32)
    return out

def ensemble_predict_proba(model_dict, X):
    probs = []
    for _, model in model_dict.items():
        probs.append(model.predict_proba(X)[:, 1])
    return np.mean(np.column_stack(probs), axis=1)

# ─────────────────────────────────────────────────────────────────────────────
# Batch prediction helpers
# ─────────────────────────────────────────────────────────────────────────────
# IMPORTANT: predict_proba scores each row (pixel) from its OWN feature vector.
# There is NO spatial interpolation or extrapolation between pixels.
# Batching is purely a RAM management strategy.
# The concatenated result is bit-for-bit identical to predict_proba(X)[:,1]
# on the full array at once.
PRED_BATCH_SIZE = 200_000  # rows per chunk; reduce to 50_000 if RAM is tight

def predict_in_batches(model, X, batch_size=PRED_BATCH_SIZE):
    """Each pixel is scored from its own row of features.
    Batch boundaries have ZERO influence on any individual prediction.
    Output is identical to model.predict_proba(X)[:, 1].
    """
    n = len(X)
    out = np.empty(n, dtype=np.float32)
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        out[start:end] = model.predict_proba(X.iloc[start:end])[:, 1]
    return out


def domain_score_in_batches(X, ad_model, batch_size=PRED_BATCH_SIZE):
    """Batch wrapper for compute_domain_score — same pixel-independence guarantee."""
    n = len(X)
    scores = np.empty(n, dtype=np.float32)
    dists  = np.empty(n, dtype=np.float32)
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        s, d = compute_domain_score(X.iloc[start:end], ad_model)
        scores[start:end] = s
        dists[start:end]  = d
    return scores, dists



# =========================================================
# 3. APPLICABILITY DOMAIN
# =========================================================
def fit_applicability_domain(X_train):
    """
    Fits a feature-space trust model using Mahalanobis distance.
    LedoitWolf is used for stable covariance estimation.
    """
    X_train = X_train.copy()

    mean_vec = X_train.mean(axis=0).values.astype(np.float64)
    cov_est = LedoitWolf().fit(X_train.values.astype(np.float64))

    precision = cov_est.precision_
    train_centered = X_train.values.astype(np.float64) - mean_vec

    train_dist = np.sqrt(np.einsum("ij,jk,ik->i", train_centered, precision, train_centered))

    # 95th percentile threshold from training samples
    threshold = np.percentile(train_dist, 95)

    return {
        "mean": mean_vec,
        "precision": precision,
        "threshold": float(threshold)
    }

def compute_domain_score(X_pixels, ad_model):
    """
    Score in [0,1]
    1 = very close to training feature space
    0 = far outside training feature space
    """
    Xv = X_pixels.values.astype(np.float64)
    centered = Xv - ad_model["mean"]
    dist = np.sqrt(np.einsum("ij,jk,ik->i", centered, ad_model["precision"], centered))

    threshold = ad_model["threshold"]

    # score = 1 at distance 0, score drops to 0 when distance >= 2 * threshold
    score = 1.0 - np.clip(dist / (2.0 * threshold), 0, 1)
    return score.astype(np.float32), dist.astype(np.float32)

def compute_agreement_score(prob_list):
    """
    Agreement score in [0,1]
    Higher = models agree more
    """
    probs = np.column_stack(prob_list)   # [n_samples, n_models]
    std = np.std(probs, axis=1)

    # max std in [0,1] probs is around 0.5
    agreement = 1.0 - np.clip(std / 0.5, 0, 1)
    return agreement.astype(np.float32), std.astype(np.float32)

# =========================================================
# 4. MAIN LOOP
# =========================================================
# for grid in ["3x3", "15x15", "25x25"]:
for grid in excel_files:   # only processes grids whose entry is uncommented above
    print(f"\n{'='*25} GRID: {grid} {'='*25}")

    grid_out_dir = os.path.join(output_base_dir, grid)
    make_dir(grid_out_dir)
    make_dir(os.path.join(grid_out_dir, "maps"))
    make_dir(os.path.join(grid_out_dir, "pngs"))

    # -----------------------------------------------------
    # Load models first so feature_names is known, then load
    # training data using exactly those features.
    # -----------------------------------------------------
    rf_model, svm_model, xgb_model, ensemble_bundle = load_models_for_grid(grid)

    feature_names = ensemble_bundle["features"]
    print("Feature order used by models:")
    print(feature_names)

    X_train, y_train = load_training_features(grid, feature_names, ensemble_bundle)

    # -----------------------------------------------------
    # Load raster stack
    # -----------------------------------------------------
    stack, profile, transform, crs, nodata_ref, raster_paths, bounds_ref = load_raster_stack(grid, feature_names)
    valid_mask = build_valid_mask(stack, nodata_ref=nodata_ref)

    rows, cols = valid_mask.shape
    X_pixels = stack_to_dataframe(stack, valid_mask, feature_names)

    print("Raster shape:", (rows, cols))
    print("Valid pixels:", len(X_pixels))

    # -----------------------------------------------------
    # Predict probabilities
    # -----------------------------------------------------
    # Each pixel scored from its own features — NO interpolation.
    # predict_in_batches is RAM management only; output == predict_proba(X)[:,1].
    print(f"  RF  ({len(X_pixels):,} pixels, batch_size={PRED_BATCH_SIZE:,})...")
    rf_prob_valid  = predict_in_batches(rf_model,  X_pixels)
    print("  SVM...")
    svm_prob_valid = predict_in_batches(svm_model, X_pixels)
    print("  XGB...")
    xgb_prob_valid = predict_in_batches(xgb_model, X_pixels)

    ensemble_prob_valid = np.mean(
        np.column_stack([rf_prob_valid, svm_prob_valid, xgb_prob_valid]),
        axis=1
    )

    # -----------------------------------------------------
    # Applicability domain
    # -----------------------------------------------------
    ad_model = fit_applicability_domain(X_train)
    print("  Domain score (batched)...")
    domain_score_valid, mahal_dist_valid = domain_score_in_batches(X_pixels, ad_model)

    agreement_score_valid, disagreement_std_valid = compute_agreement_score(
        [rf_prob_valid, svm_prob_valid, xgb_prob_valid]
    )

    # Combined applicability / trust score
    applicability_valid = 0.7 * domain_score_valid + 0.3 * agreement_score_valid

    # Binary trusted mask
    trusted_valid = (applicability_valid >= 0.5).astype(np.float32)

    # -----------------------------------------------------
    # Reshape to rasters
    # -----------------------------------------------------
    rf_map = reshape_predictions(rf_prob_valid, valid_mask, (rows, cols))
    svm_map = reshape_predictions(svm_prob_valid, valid_mask, (rows, cols))
    xgb_map = reshape_predictions(xgb_prob_valid, valid_mask, (rows, cols))
    ensemble_map = reshape_predictions(ensemble_prob_valid, valid_mask, (rows, cols))

    domain_score_map = reshape_predictions(domain_score_valid, valid_mask, (rows, cols))
    agreement_score_map = reshape_predictions(agreement_score_valid, valid_mask, (rows, cols))
    applicability_map = reshape_predictions(applicability_valid, valid_mask, (rows, cols))
    trusted_map = reshape_predictions(trusted_valid, valid_mask, (rows, cols))
    disagreement_std_map = reshape_predictions(disagreement_std_valid, valid_mask, (rows, cols))
    mahal_dist_map = reshape_predictions(mahal_dist_valid, valid_mask, (rows, cols))

    # -----------------------------------------------------
    # Save GeoTIFFs
    # -----------------------------------------------------
    write_raster(os.path.join(grid_out_dir, "maps", f"RF_susceptibility_{grid}.tif"), rf_map, profile)
    write_raster(os.path.join(grid_out_dir, "maps", f"SVM_susceptibility_{grid}.tif"), svm_map, profile)
    write_raster(os.path.join(grid_out_dir, "maps", f"XGB_susceptibility_{grid}.tif"), xgb_map, profile)
    write_raster(os.path.join(grid_out_dir, "maps", f"Ensemble_susceptibility_{grid}.tif"), ensemble_map, profile)

    write_raster(os.path.join(grid_out_dir, "maps", f"Domain_score_{grid}.tif"), domain_score_map, profile)
    write_raster(os.path.join(grid_out_dir, "maps", f"Agreement_score_{grid}.tif"), agreement_score_map, profile)
    write_raster(os.path.join(grid_out_dir, "maps", f"Applicability_score_{grid}.tif"), applicability_map, profile)
    write_raster(os.path.join(grid_out_dir, "maps", f"Trusted_mask_{grid}.tif"), trusted_map, profile)
    write_raster(os.path.join(grid_out_dir, "maps", f"Model_disagreement_std_{grid}.tif"), disagreement_std_map, profile)
    write_raster(os.path.join(grid_out_dir, "maps", f"Mahalanobis_distance_{grid}.tif"), mahal_dist_map, profile)

    # -----------------------------------------------------
    # Save PNG previews
    # -----------------------------------------------------
    quick_plot(rf_map, f"RF Susceptibility - {grid}", os.path.join(grid_out_dir, "pngs", f"RF_susceptibility_{grid}.png"), cmap="RdYlBu_r", bounds=bounds_ref)
    quick_plot(svm_map, f"SVM Susceptibility - {grid}", os.path.join(grid_out_dir, "pngs", f"SVM_susceptibility_{grid}.png"), cmap="RdYlBu_r", bounds=bounds_ref)
    quick_plot(xgb_map, f"XGB Susceptibility - {grid}", os.path.join(grid_out_dir, "pngs", f"XGB_susceptibility_{grid}.png"), cmap="RdYlBu_r", bounds=bounds_ref)
    quick_plot(ensemble_map, f"Ensemble Susceptibility - {grid}", os.path.join(grid_out_dir, "pngs", f"Ensemble_susceptibility_{grid}.png"), cmap="RdYlBu_r", bounds=bounds_ref)

    quick_plot(domain_score_map, f"Domain Score - {grid}", os.path.join(grid_out_dir, "pngs", f"Domain_score_{grid}.png"), cmap="viridis", bounds=bounds_ref)
    quick_plot(agreement_score_map, f"Agreement Score - {grid}", os.path.join(grid_out_dir, "pngs", f"Agreement_score_{grid}.png"), cmap="viridis", bounds=bounds_ref)
    quick_plot(applicability_map, f"Applicability Score - {grid}", os.path.join(grid_out_dir, "pngs", f"Applicability_score_{grid}.png"), cmap="viridis", bounds=bounds_ref)
    quick_plot(trusted_map, f"Trusted Prediction Mask - {grid}", os.path.join(grid_out_dir, "pngs", f"Trusted_mask_{grid}.png"), cmap="gray", bounds=bounds_ref)
    quick_plot(disagreement_std_map, f"Model Disagreement STD - {grid}", os.path.join(grid_out_dir, "pngs", f"Model_disagreement_std_{grid}.png"), cmap="magma", bounds=bounds_ref)

    # -----------------------------------------------------
    # Journal-quality composite + classified maps
    # -----------------------------------------------------
    plot_multi_model_maps(
        {"RF": rf_map, "SVM": svm_map, "XGB": xgb_map, "Ensemble": ensemble_map},
        bounds=bounds_ref, grid_name=grid,
        save_path=os.path.join(grid_out_dir, "pngs", f"Multi_model_susceptibility_{grid}.png"),
    )

    _pixel_area_m2 = abs(profile["transform"].a * profile["transform"].e)
    areal_df = plot_classified_susceptibility(
        ensemble_map, bounds=bounds_ref, grid_name=grid,
        save_path=os.path.join(grid_out_dir, "pngs", f"Classified_susceptibility_{grid}.png"),
        pixel_area_m2=_pixel_area_m2,
    )
    areal_df.to_csv(os.path.join(grid_out_dir, f"{grid}_areal_statistics.csv"), index=False)

    # -----------------------------------------------------
    # Save metadata / summary
    # -----------------------------------------------------
    summary = {
        "grid": grid,
        "feature_names": feature_names,
        "n_valid_pixels": int(len(X_pixels)),
        "training_samples": int(len(X_train)),
        "applicability_threshold_binary_mask": 0.5,
        "applicability_formula": "0.7 * domain_score + 0.3 * agreement_score",
        "domain_threshold_95pct_train_mahalanobis": float(ad_model["threshold"]),
        "maps_saved_in": os.path.join(grid_out_dir, "maps")
    }

    with open(os.path.join(grid_out_dir, f"{grid}_prediction_summary.json"), "w") as f:
        json.dump(summary, f, indent=4)

    print(f"Finished grid: {grid}")
    print(json.dumps(summary, indent=2))

    gc.collect()

print("\nAll susceptibility, ensemble, and applicability maps created.")


Output hidden; open in https://colab.research.google.com to view.